# UN Sentiment & Tone Analysis

This analysis examines sentiment and tone in UN Noon Briefings by classifying briefing sections as Positive, Neutral, or Negative and identifying their communicative tone, such as Diplomatic, Concerned, Critical, Urgent, Optimistic, Neutral/Informational, or Constructive. 

# 1. Set up and load data
This notebook uses the pre-uploaded database specified by BRIEFINGS_DB_PATH.

Alternatively, the database can be created from scratch for a desired date range:

briefings update 2026-09-01 2026-09-10

briefings init

briefings load 2026-09-01 2026-09-10

In [10]:
import pandas as pd
import ollama
from tqdm import tqdm

from un_llm_analysis.config import BASE_URL, BRIEFINGS_DB_PATH
from un_llm_analysis.database import (
    get_db_connection, 
    get_briefing_id_from_date, 
    get_briefings_between_dates,
    insert_into_analysis_run,
    insert_sentiment_analysis
)

In [11]:
# Get the data in the database with its maximum date range
db_start_date = "2021-01-01"
db_end_date = "2026-09-10"

with get_db_connection(BRIEFINGS_DB_PATH) as conn:
    briefings = get_briefings_between_dates(conn, db_start_date, db_end_date)

In [12]:
# Checking the content in the database
from un_llm_analysis.database import get_db_connection

conn = get_db_connection()

try:
    cursor = conn.cursor()

    tables = [
        "briefing",
        "section",
        "analysis_run",
        "sentiment_analysis",
    ]

    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"{table}: {count} records")

finally:
    conn.close()

briefing: 10 records
section: 88 records
analysis_run: 19 records
sentiment_analysis: 1626 records


In [13]:
# Get the section data from the briefings and store it in a DataFrame

rows = []

for briefing in tqdm(briefings):
    briefing_id = get_briefing_id_from_date(briefing.briefing_date)
    for i, section in enumerate(briefing.sections, start=1):
        section_id = briefing_id * 100 + i
        rows.append(
            {
                "section_id": section_id,
                "briefing_id": briefing_id,
                "briefing_date": briefing.briefing_date,
                "section_title": section.title,
                "section_text": section.text,
            }
        )

section_data = pd.DataFrame(rows)

section_data.info()
section_data.head(2)

100%|██████████| 10/10 [00:00<00:00, 5327.45it/s]


<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     88 non-null     int64
 1   briefing_id    88 non-null     int64
 2   briefing_date  88 non-null     str  
 3   section_title  88 non-null     str  
 4   section_text   88 non-null     str  
dtypes: int64(2), str(3)
memory usage: 102.7 KB


,section_id,briefing_id,briefing_date,section_title,section_text
0,2026090101,20260901,2026-09-01,TODAY’S BRIFING,"At 12:30 p.m., Jérôme Bonnafont, the Permanent..."
1,2026090102,20260901,2026-09-01,TOMORROW’S GUEST,"Inger Andersen, the Executive Director of UNEP..."


In [14]:
# Prompt v3: complete labels and explicit classification rules
prompt_version = "v4"

VALID_SENTIMENTS = {"Positive", "Neutral", "Negative"}
VALID_TONES = {
    "Diplomatic",
    "Concerned",
    "Critical",
    "Urgent",
    "Optimistic",
    "Neutral/Informational",
    "Constructive",
}

ANALYSIS_PROMPT = """
You are classifying a section from a United Nations Noon Briefing.

Classify SENTIMENT using exactly one label:
- Positive: favorable developments, progress, success, support, or hope.
- Neutral: factual reporting without clearly positive or negative evaluation.
- Negative: adverse developments, harm, criticism, failure, or deterioration.

Classify the PRIMARY TONE using exactly one label:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational
- Constructive

Rules:
- Sentiment and tone are different. A negative situation can be reported neutrally.
- Do not label text Negative only because it mentions conflict or a crisis.
- Base both labels on the actual wording and overall message.
- Choose the single most prominent tone.
- Do not infer emotions that are not supported by the text.

Return only a valid JSON object with exactly these keys:
{"sentiment": "Neutral", "tone": "Neutral/Informational"}

SECTION TITLE:
__SECTION_TITLE__

SECTION TEXT:
__SECTION_TEXT__
""".strip()

print(f"Prompt version: {prompt_version}")
print(ANALYSIS_PROMPT)


Prompt version: v4
You are classifying a section from a United Nations Noon Briefing.

Classify SENTIMENT using exactly one label:
- Positive: favorable developments, progress, success, support, or hope.
- Neutral: factual reporting without clearly positive or negative evaluation.
- Negative: adverse developments, harm, criticism, failure, or deterioration.

Classify the PRIMARY TONE using exactly one label:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational
- Constructive

Rules:
- Sentiment and tone are different. A negative situation can be reported neutrally.
- Do not label text Negative only because it mentions conflict or a crisis.
- Base both labels on the actual wording and overall message.
- Choose the single most prominent tone.
- Do not infer emotions that are not supported by the text.

Return only a valid JSON object with exactly these keys:
{"sentiment": "Neutral", "tone": "Neutral/Informational"}

SECTION TITLE:
__SECTION_TITLE__

SECTION T

#Install the required Python packages

pipenv install mlflow openai

#Open a second terminal in your project folder and run MLflow server

pipenv run mlflow server --host 127.0.0.1 --port 5000



In [16]:
import mlflow
from openai import OpenAI

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("sentiment and tone analysis")

# Ollama's OpenAI-compatible API. The API key is required by the SDK,
# but Ollama does not validate this placeholder key.
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [17]:
# This cell is running for sentiment and tone analysis of sections of UN noon briefing

import json
import mlflow
import pandas as pd

models = ["llama3.2"]
analysis_name = "UN briefing sentiment and tone analysis"
temperature = 0
seed = 42
max_tokens = 256

# Set this before entering the try block so a failed run creation does not
# leave a misleading, undefined analysis_id in later notebook cells.
analysis_id = None
conn = get_db_connection()

try:
    for llm in models:
        analysis_id = insert_into_analysis_run(
            conn=conn,
            analysis_name=analysis_name,
            model_name=llm,
            model_version=llm,
            ollama_version=None,
            prompt_version=prompt_version,
            prompt=ANALYSIS_PROMPT,
            temperature=temperature,
        )

        success_count = 0
        error_count = 0

        with mlflow.start_run(run_name=f"{llm}-{prompt_version}"):
            mlflow.log_params({
                "model": llm,
                "prompt_version": prompt_version,
                "temperature": temperature,
                "seed": seed,
                "max_tokens": max_tokens,
                "n_sections": len(section_data),
            })

            for _, section in tqdm(
                section_data.iterrows(),
                total=len(section_data),
                desc=f"Classifying with {llm}",
            ):
                sentiment = None
                tone = None
                status = "error"
                error_message = None

                # Keep null text explicit rather than classifying "nan".
                title_value = section.get("section_title")
                text_value = section.get("section_text")
                section_title = (
                    "" if pd.isna(title_value) else str(title_value).strip()
                )
                section_text = (
                    "" if pd.isna(text_value) else str(text_value).strip()
                )

                try:
                    if not section_text:
                        raise ValueError("Section text is empty.")

                    # Use literal placeholder replacement. str.replace() does
                    # not accept section_text=... as a named argument.
                    prompt = (
                        ANALYSIS_PROMPT
                        .replace("__SECTION_TITLE__", section_title or "(no title)")
                        .replace("__SECTION_TEXT__", section_text)
                    )

                    response = client.chat.completions.create(
                        model=llm,
                        messages=[
                            {
                                "role": "user",
                                "content": prompt,
                            }
                        ],
                        temperature=temperature,
                        seed=seed,
                        max_tokens=max_tokens,
                        response_format={"type": "json_object"},
                    )

                    result_text = response.choices[0].message.content
                    if not result_text:
                        raise ValueError("The model returned an empty response.")

                    result = json.loads(result_text)
                    if not isinstance(result, dict):
                        raise ValueError("Model response must be a JSON object.")

                    sentiment = result.get("sentiment")
                    tone = result.get("tone")

                    # Exact matching deliberately rejects spelling/case variants.
                    if sentiment not in VALID_SENTIMENTS:
                        raise ValueError(
                            f"Invalid sentiment label: {sentiment!r}. "
                            f"Expected one of {sorted(VALID_SENTIMENTS)}"
                        )
                    if tone not in VALID_TONES:
                        raise ValueError(
                            f"Invalid tone label: {tone!r}. "
                            f"Expected one of {sorted(VALID_TONES)}"
                        )

                    status = "success"
                    success_count += 1

                except Exception as exc:
                    error_message = f"{type(exc).__name__}: {exc}"
                    error_count += 1
                    print(
                        f"[ERROR] analysis_id={analysis_id}, "
                        f"section_id={section.get('section_id')}: "
                        f"{error_message}"
                    )

                # Persist every section, including failures, so they can be
                # identified and retried instead of silently disappearing.
                insert_sentiment_analysis(
                    conn=conn,
                    analysis_id=analysis_id,
                    briefing_id=int(section.briefing_id),
                    section_id=int(section.section_id),
                    briefing_date=section.get("briefing_date"),
                    sentiment=sentiment,
                    tone=tone,
                    status=status,
                    error_message=error_message,
                )

            mlflow.log_metrics({
                "successful_sections": success_count,
                "failed_sections": error_count,
            })

        print(
            f"Analysis run {analysis_id} finished: "
            f"{success_count} successful, {error_count} failed."
        )

finally:
    conn.close()


Classifying with llama3.2: 100%|██████████| 88/88 [15:00<00:00, 10.23s/it]


🏃 View run llama3.2-v4 at: http://127.0.0.1:5000/#/experiments/4/runs/e2295ae4bf254e6eaa93c49f7f68a9fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Analysis run 20 finished: 88 successful, 0 failed.


# Reference



1.This notebook is based on the following notebook.
[Sentiment Analysis Issue Mapping  Notebook](notebooks/franz/02_analyze_issue_mapping.ipynb).
